# Distribution Planning Mean-Field Control Benchmark

Ten-state torus benchmark where the planner moves population mass toward a target law while paying movement cost. The article use is a larger discrete state-space stress test for flow control and budget scaling.

## Article Figure Set

This notebook is organized around the six exhibits that belong in the paper:

1. Main learning comparison: validation objective over training for simplex, logit MFREINFORCE, and REINFORCE.
2. Final performance table: objective, gap to the known optimum when available, policy/flow error when available, runtime, and budget.
3. Initial/target/final distribution comparison: grouped probability bars by state index, matching the Meunier-style distribution-planning figure.
4. Perturbation scale diagnostics: policy error, $J$ vs $J^\lambda$, and perturbation coverage.
5. Gradient decomposition: bias, variance, MSE, and the missing mean-field term exposed by the REINFORCE ablation.
6. Horizon and flow robustness: performance/runtime as $T$ and the nominal-flow estimator change.

Generalization checks are intentionally left for appendix-style follow-up cells; they are useful, but not central to the paper's claim.

In [ ]:

import sys
import time
from pathlib import Path

_notebook_start = time.perf_counter()

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
for path in (SRC, ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import pandas as pd
import torch

# These diagnostics are small-tensor heavy; CPU is usually clearer and faster here.
torch.set_default_device("cpu")
torch.set_default_dtype(torch.float64)

from configs.distribution_planning import MAIN, MID, SMOKE
from mfc.environments.distribution_planning import DistributionPlanning
from mfc.algorithms import reinforce as reinforce_alg
from mfc.algorithms import simplex
from mfc.plotting import diagnostics as viz
from mfc.plotting.style import apply_style, color_for, set_style, style_legend
from scripts.train import run_all
from scripts.test import (
    constant_policy_fn,
    exact_gradient,
    group_by,
    load_runs,
    logit_perturbation_coverage,
    objective_gap,
    perturbation_coverage,
    policy_error,
    population_tracking_error,
    state_distribution,
)

set_style()

ENV_NAME = "distribution_planning"
ENV_CLASS = DistributionPlanning
REF_LAM = 0.2
DIAGNOSTIC_REPS = 25
OBJECTIVE_SAMPLES = 1000
COVERAGE_SAMPLES = 5000
VALIDATION_MU0_SAMPLES = 32


## Configuration and Data

Load the richest cached tier available: `main`, then `mid`, then `smoke`. Smoke runs are trained inline only when no cache exists.

In [ ]:

def cached_count(tier_name):
    run_dir = ROOT / "runs" / ENV_NAME / tier_name
    return len(list(run_dir.glob("*_seed*.pt")))

for tier_name, candidate in (("main", MAIN), ("mid", MID), ("smoke", SMOKE)):
    if cached_count(tier_name):
        tier, cfg = tier_name, candidate
        break
else:
    tier, cfg = "smoke", SMOKE

runs_dir = ROOT / "runs" / ENV_NAME / tier
for alg in cfg.algorithms:
    if tier == "smoke" and not list(runs_dir.glob(f"{alg}_*_seed*.pt")):
        run_all(ENV_NAME, alg, "smoke", progress="off")

runs = []
for alg in cfg.algorithms:
    runs.extend(load_runs(ENV_NAME, alg, tier, device="cpu"))

if not runs:
    raise RuntimeError(f"No runs found for {ENV_NAME}/{tier}; run scripts/train.py first.")

run_dtype = runs[0]["theta_final"].dtype
torch.set_default_dtype(run_dtype)
env = ENV_CLASS(dtype=run_dtype, device="cpu")
mu0_val = torch.tensor(cfg.mu0_val, dtype=env.dtype, device="cpu")
gamma = getattr(env.config, "gamma", 1.0)

def _matches(r, *, budget_mode=None, flow=None, T=None, lam="any", alg=None):
    return ((budget_mode is None or r["budget_mode"] == budget_mode)
            and (flow is None or r["flow"] == flow)
            and (T is None or r["T"] == T)
            and (lam == "any" or r["lam"] == lam)
            and (alg is None or r["alg"] == alg))

def _closest_lambda(available, target):
    return min(available, key=lambda x: abs(float(x) - float(target)))

simplex_runs = [r for r in runs if r["alg"] == "simplex" and r["lam"] is not None]
available_lambdas = sorted({r["lam"] for r in simplex_runs})
ref_lam = _closest_lambda(available_lambdas, REF_LAM) if available_lambdas else None

default_T = cfg.horizons[0]
default_reference_key = (cfg.budget_modes[0], cfg.flows[0], default_T, ref_lam)
candidate_groups = [
    default_reference_key,
    ("equal_budget", "particle", default_T, ref_lam),
    ("equal_parameters", "exact", default_T, ref_lam),
    (cfg.budget_modes[0], cfg.flows[0], None, ref_lam),
]
reference_key = None
for bm, fl, T, lam in candidate_groups:
    matched = [r for r in runs if _matches(r, budget_mode=bm, flow=fl, T=T, lam=lam, alg="simplex")]
    if matched:
        chosen = matched[0]
        reference_key = (chosen["budget_mode"], chosen["flow"], chosen["T"], chosen["lam"])
        break
if reference_key is None:
    chosen = simplex_runs[0] if simplex_runs else runs[0]
    reference_key = (chosen["budget_mode"], chosen["flow"], chosen["T"], chosen["lam"])

ref_budget_mode, ref_flow, ref_T, ref_lam = reference_key
if reference_key != default_reference_key:
    print(
        "warning: no cached runs for the config default group "
        f"budget_mode={default_reference_key[0]}, flow={default_reference_key[1]}, T={default_reference_key[2]}, lambda={default_reference_key[3]}; "
        "falling back to an available cached group. Regenerate the cache to reproduce the reference protocol."
    )
comparison_runs = [r for r in runs if _matches(r, budget_mode=ref_budget_mode, flow=ref_flow, T=ref_T)]
comparison_simplex = [r for r in comparison_runs if r["alg"] == "simplex" and r["lam"] == ref_lam]
reference_run = sorted(comparison_simplex or comparison_runs, key=lambda r: (r["seed"], r["alg"]))[0]

def batch_size_for(alg, T, budget_mode):
    if alg == "simplex":
        return cfg.simplex_B_equal_budget(T) if budget_mode == "equal_budget" else cfg.B
    if alg == "reinforce":
        return cfg.reinforce_B_equal_budget(T) if budget_mode == "equal_budget" else cfg.B
    return cfg.B

def simplex_n_aux_for():
    return getattr(cfg, "simplex_n_aux", cfg.n_aux)

def aux_size_for(alg):
    if alg == "simplex":
        return simplex_n_aux_for()
    if alg == "mfreinforce":
        return cfg.n_aux
    return 0

def transitions_per_step_for(alg, T, budget_mode):
    if budget_mode == "equal_budget":
        return cfg.equal_budget_target(T)
    if alg == "mfreinforce":
        return cfg.logit_transitions(T) // T
    return batch_size_for(alg, T, budget_mode) + aux_size_for(alg)

optimal_theta = env.optimal_theta() if hasattr(env, "optimal_theta") else None
optimal_J = None
if optimal_theta is not None:
    optimal_J = simplex.exact_objective(env, env.policy_probs, optimal_theta, mu0_val, ref_T, gamma=gamma).item()

print(f"tier: {tier}")
print(f"loaded runs: {len(runs)}")
print(f"article reference group: budget_mode={ref_budget_mode}, flow={ref_flow}, T={ref_T}, lambda={ref_lam}")
print(f"total cached training time: {sum(r['elapsed_seconds'] for r in runs):.1f}s")
group_keys = sorted(group_by(runs, 'budget_mode', 'flow', 'T', 'lam'), key=lambda k: (k[0], k[1], k[2], -1.0 if k[3] is None else float(k[3])))
print(f"available groups: {group_keys[:8]}{' ...' if len(group_keys) > 8 else ''}")

VALIDATION_MU0_SAMPLES = 32
VALIDATION_MU0_SEED = 2025
MAX_MEUNIER_CHECKPOINTS = 80


def validation_mu0_set(n=VALIDATION_MU0_SAMPLES):
    """Fixed validation set for Meunier-style V(mu_0) curves."""
    generator = torch.Generator(device="cpu").manual_seed(VALIDATION_MU0_SEED)
    if hasattr(env, "sample_mu0"):
        return env.sample_mu0((n,), generator=generator).detach().cpu()
    return mu0_val.detach().cpu().unsqueeze(0)


def _history_subset(run, max_points=MAX_MEUNIER_CHECKPOINTS):
    if "theta_history" not in run:
        raise RuntimeError(
            "Meunier-style V(mu_0) training curves need theta_history checkpoints. "
            "Re-run the cached training jobs with the current scripts/train.py, "
            "or remove the old cache files so the smoke tier can be regenerated."
        )
    history = run["theta_history"].to(dtype=env.dtype, device="cpu")
    if "theta_history_iterations" in run:
        iterations = run["theta_history_iterations"].to(device="cpu")
    elif "validation_iterations" in run and run["validation_iterations"].shape[0] == history.shape[0]:
        iterations = run["validation_iterations"].to(device="cpu")
    else:
        n_train = getattr(run["config"], "n_train", None)
        if n_train is not None:
            iterations = torch.linspace(0, n_train, history.shape[0]).round().to(torch.long)
        else:
            iterations = torch.arange(history.shape[0], dtype=torch.long)
    if history.shape[0] <= max_points:
        return iterations, history
    idx = torch.linspace(0, history.shape[0] - 1, max_points).round().to(torch.long)
    idx = torch.unique_consecutive(idx)
    if idx[-1].item() != history.shape[0] - 1:
        idx = torch.cat([idx, torch.tensor([history.shape[0] - 1])])
    return iterations[idx], history[idx]


def value_curve_over_mu0(run, mu0_grid):
    """
    Evaluate V(theta_m; mu_0) at stored theta checkpoints over a fixed
    validation set of initial laws. Returns values with shape
    (n_checkpoints, n_mu0).
    """
    iterations, history = _history_subset(run)
    T_val = getattr(run["config"], "T_val", None) or run["T"]
    values = []
    with torch.no_grad():
        for theta in history:
            row = [simplex.exact_objective(env, env.policy_probs, theta, mu0.to(dtype=env.dtype), T_val, gamma=gamma) for mu0 in mu0_grid]
            values.append(torch.stack(row).detach().cpu())
    return {"iterations": iterations, "values": torch.stack(values)}


def plot_v_mu0_validation_curve(runs, *, mu0_grid=None, ax=None):
    """
    Meunier-style validation curve. For each run, compute V(theta_m; mu_0)
    on a fixed validation set. The line averages over validation initial laws
    and training seeds; the shaded band is the std over validation initial
    laws, averaged across seeds. It is not the seed-to-seed band from the
    cached validation_J plot.
    """
    if mu0_grid is None:
        mu0_grid = validation_mu0_set()
    fig, ax = (ax.figure, ax) if ax is not None else plt.subplots(figsize=(6.0, 4.0))
    groups = group_by(runs, "alg", "lam")
    several_algs = len({alg for alg, _ in groups}) > 1
    for (alg, lam), group in sorted(groups.items(), key=lambda kv: (kv[0][1] is None, kv[0][1], kv[0][0])):
        curves = [value_curve_over_mu0(r, mu0_grid) for r in group]
        # Runs in one group share the same saved history schedule; trim to the
        # shortest defensive length if older cached files differ.
        length = min(c["values"].shape[0] for c in curves)
        iterations = curves[0]["iterations"][:length]
        values = torch.stack([c["values"][:length] for c in curves])  # (seed, checkpoint, mu0)
        per_seed_mean = values.mean(dim=-1)
        per_seed_std = values.std(dim=-1, unbiased=False)
        mean = per_seed_mean.mean(dim=0)
        std_mu0 = per_seed_std.mean(dim=0)
        label = alg if lam is None else (f"{alg} λ={lam}" if several_algs else f"λ={lam}")
        (line,) = ax.plot(iterations, mean, label=label)
        ax.fill_between(iterations, mean - std_mu0, mean + std_mu0, color=line.get_color(), alpha=0.2)
    apply_style(ax, xlabel="training iteration", ylabel=r"validation $V(\mu_0)$")
    style_legend(ax)
    return fig, ax


## 1. Main Learning Comparison

Validation reward over training in the Meunier style: each curve is the cached exact value $V(\mu_0)$ at the benchmark validation initial law, evaluated throughout training.

The line is the mean over five independent training seeds. The shaded band is **mean ± one standard deviation over those seeds**, matching the Meunier validation-reward plots. Simplex is shown for each cached $\lambda$; MFREINFORCE and REINFORCE each appear as their own non-$\lambda$ baselines.

Distribution planning is optimized as a reward-maximization problem. Rewards are negative costs, so values are typically non-positive and **higher is better**.

In [ ]:
fig, ax = viz.plot_validation_curve(comparison_runs)
ax.set_ylabel(r"$V(\mu_0)$ (Mean ± Std. Dev.)")
ax.set_title("Mean Validation Reward vs. Training Steps")

## 2. Final Performance Table

A compact article table: final validation $V(\mu_0^\mathrm{val})$ mean/std over training seeds, plus tracking diagnostics, runtime, and the effective batch budget.

In [ ]:
rows = []
for (alg, lam), group in sorted(group_by(comparison_runs, "alg", "lam").items(), key=lambda kv: (kv[0][0], -1 if kv[0][1] is None else kv[0][1])):
    final_J = torch.tensor([r["validation_J"][-1].item() for r in group], dtype=torch.float64)
    flow_errors = []
    policy_errors = []
    for r in group:
        theta = r["theta_final"].to(dtype=env.dtype, device="cpu")
        flow_errors.append(population_tracking_error(env, env.policy_probs, theta, mu0_val, r["T"]).item() if hasattr(env, "target_law") else float("nan"))
        if hasattr(env, "optimal_policy"):
            policy_errors.append(policy_error(env, env.policy_probs, theta).mean().item())

    row = {
        "algorithm": alg,
        "lambda": "-" if lam is None else lam,
        "seeds": len(group),
        "V(mu0) mean": final_J.mean().item(),
        "V(mu0) seed std": final_J.std(unbiased=False).item(),
        "gap to optimal": (optimal_J - final_J.mean().item()) if optimal_J is not None else float("nan"),
        "policy error": sum(policy_errors) / len(policy_errors) if policy_errors else float("nan"),
        "flow error": sum(flow_errors) / len(flow_errors) if flow_errors else float("nan"),
        "runtime mean s": sum(r["elapsed_seconds"] for r in group) / len(group),
        "B used": batch_size_for(alg, ref_T, ref_budget_mode),
        "n_aux used": aux_size_for(alg),
        "transitions/step": transitions_per_step_for(alg, ref_T, ref_budget_mode),
    }
    rows.append(row)

performance_table = pd.DataFrame(rows)
performance_table["_lambda_sort"] = performance_table["lambda"].map(lambda x: -1.0 if x == "-" else float(x))
performance_table = performance_table.sort_values(["algorithm", "_lambda_sort"]).drop(columns="_lambda_sort")
performance_table

## 3. Initial, Target, and Final Distribution

Meunier-style grouped bar chart over state index: the fixed validation initial law, the target law, and the terminal distribution produced by the selected learned policy. This replaces the per-time learned mean-field-flow plot, which was too busy for this benchmark.

In [ ]:
theta_hat = reference_run["theta_final"]
mu_flow = state_distribution(env, env.policy_probs, theta_hat, mu0_val, ref_T)
final_law = mu_flow[-1]

fig, ax = viz.plot_distribution_comparison(
    {
        "initial": mu0_val,
        "target": env.target_law,
        "final": final_law,
    },
    state_labels=[str(i) for i in range(env.n_states)],
)
ax.set_title(
    f"{ENV_NAME}: initial, target, final distribution "
    f"({reference_run['alg']}, lambda={reference_run['lam']}, seed={reference_run['seed']})"
)
fig

## 4. Perturbation Scale Diagnostics

The λ sweep should show the expected trade-off: smaller perturbations reduce objective bias but raise estimator variance. The coverage table checks the simplex guarantee $d_{TV}(M^\lambda,\mu)\leq\lambda$ on representative laws.

In [ ]:

lambda_runs = {}
for lam in sorted({r["lam"] for r in comparison_runs if r["alg"] == "simplex" and r["lam"] is not None}):
    candidates = [r for r in comparison_runs if r["alg"] == "simplex" and r["lam"] == lam]
    if candidates:
        lambda_runs[lam] = sorted(candidates, key=lambda r: r["seed"])[0]

if not lambda_runs:
    print("No simplex lambda sweep found for this group.")
else:
    fig, axes = plt.subplots(1, 2 if hasattr(env, "optimal_policy") else 1, figsize=(10, 3.6), constrained_layout=True)
    axes = axes if isinstance(axes, (list, tuple)) or hasattr(axes, "__len__") else [axes]

    gaps = {
        lam: objective_gap(env, env.policy_probs, r["theta_final"], mu0_val, r["T"], lam=lam, sigma=cfg.sigma,
                           n_samples=OBJECTIVE_SAMPLES, gamma=gamma, generator=torch.Generator(device="cpu").manual_seed(100 + int(1000 * lam)))
        for lam, r in lambda_runs.items()
    }
    viz.plot_objective_gap(gaps, ax=axes[0])
    axes[0].set_title(r"$J^\lambda$ vs $J$ at learned $\hat\theta_\lambda$")

    if hasattr(env, "optimal_policy"):
        errors = {lam: policy_error(env, env.policy_probs, r["theta_final"]) for lam, r in lambda_runs.items()}
        viz.plot_policy_error(errors, ax=axes[1])
        axes[1].set_title("policy error vs optimal")
    fig

    laws = [mu0_val]
    if hasattr(env, "target_law"):
        laws.append(env.target_law)
    coverage = perturbation_coverage(torch.stack(laws), ref_lam, cfg.sigma, COVERAGE_SAMPLES,
                                     generator=torch.Generator(device="cpu").manual_seed(0))
    display(pd.DataFrame([{k: (v.item() if torch.is_tensor(v) and v.ndim == 0 else v) for k, v in row.items() if k != "mu"} for row in coverage]))


## 5. Gradient Decomposition and Missing Mean-Field Term

The table compares the simplex estimator against the REINFORCE ablation at the same probe point. REINFORCE keeps the direct policy-score term and drops the population-sensitivity correction, so its systematic bias is the empirical trace of the missing mean-field term.

In [ ]:

def reinforce_gradient_diagnostics(theta, *, B, reps):
    oracle = exact_gradient(env, env.policy_probs, theta, mu0_val, ref_T, gamma=gamma)
    samples = torch.stack([
        reinforce_alg.gradient_step(env, env.policy_probs, theta, mu0_val, T=ref_T, B=B, gamma=gamma,
                                    generator=torch.Generator(device="cpu").manual_seed(5000 + i))
        for i in range(reps)
    ])
    mean = samples.mean(dim=0)
    return {
        "oracle_gradient": oracle,
        "mean_estimate": mean,
        "bias": mean - oracle,
        "std": samples.std(dim=0),
        "mse": ((samples - oracle) ** 2).mean(dim=0),
    }

theta_probe = optimal_theta if optimal_theta is not None else theta_hat
simplex_diag = gradient_diagnostics(
    env, env.policy_probs, theta_probe, mu0_val, ref_T,
    lam=ref_lam, n_aux=simplex_n_aux_for(), B=batch_size_for("simplex", ref_T, ref_budget_mode), sigma=cfg.sigma,
    reps=DIAGNOSTIC_REPS, gamma=gamma, generator=torch.Generator(device="cpu").manual_seed(11),
)
reinforce_diag = reinforce_gradient_diagnostics(theta_probe, B=batch_size_for("reinforce", ref_T, ref_budget_mode), reps=DIAGNOSTIC_REPS)

gradient_table = pd.DataFrame([
    {"estimator": "simplex", "lambda": ref_lam, "bias norm": simplex_diag["bias"].norm().item(), "std norm": simplex_diag["std"].norm().item(), "mse": simplex_diag["mse"].mean().item()},
    {"estimator": "reinforce", "lambda": 0.0, "bias norm": reinforce_diag["bias"].norm().item(), "std norm": reinforce_diag["std"].norm().item(), "mse": reinforce_diag["mse"].mean().item()},
])
gradient_table


## 6. Horizon and Flow Robustness

This table/plot gathers every cached group so the paper can report horizon scaling and exact-vs-particle effects without duplicating cells across notebooks. Final values use the same fixed validation $V(\mu_0^\mathrm{val})$ history as the main learning curve.

In [ ]:
rows = []
for (alg, bm, fl, T, lam), group in sorted(group_by(runs, "alg", "budget_mode", "flow", "T", "lam").items(), key=lambda kv: (kv[0][3], kv[0][0], str(kv[0][4]), kv[0][1], kv[0][2])):
    final_J = torch.tensor([r["validation_J"][-1].item() for r in group], dtype=torch.float64)
    rows.append({
        "algorithm": alg,
        "budget": bm,
        "flow": fl,
        "T": T,
        "lambda": "-" if lam is None else lam,
        "seeds": len(group),
        "V(mu0) mean": final_J.mean().item(),
        "V(mu0) seed std": final_J.std(unbiased=False).item(),
        "runtime mean s": sum(r["elapsed_seconds"] for r in group) / len(group),
    })

scaling_table = pd.DataFrame(rows)
display(scaling_table)

fig, ax = plt.subplots(figsize=(6.8, 4.0), constrained_layout=True)
for i, ((alg, lam), sub) in enumerate(scaling_table.groupby(["algorithm", "lambda"], dropna=False)):
    if len(sub["T"].unique()) < 2:
        continue
    sub = sub.sort_values("T")
    ax.plot(sub["T"], sub["V(mu0) mean"], marker="o", color=color_for(i), label=f"{alg}, lambda={lam}")
apply_style(ax, xlabel="horizon T", ylabel=r"final validation $V(\mu_0)$")
style_legend(ax)
ax.set_title("Horizon scaling across cached runs")
fig

## Runtime

A small footer keeps notebook runtime reportable in the article artifact.

In [ ]:

print(f"Notebook wall time: {time.perf_counter() - _notebook_start:.1f}s")
print("Paper figures covered: validation, final table, distribution comparison, lambda diagnostics, gradient decomposition, horizon/flow robustness.")